# 05 · Activation checkpointing

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/nilesh-salpe/visual-ai/blob/main/build-it/distributed/05_activation_checkpointing.ipynb) &nbsp; 📖 Concept deck: [Activation memory and recomputation](https://nilesh-salpe.github.io/visual-ai/distributed/08-activation-memory/) &nbsp; ⏱️ Suggested time: **35 min**

**You will build:** a manual backward pass for a deep stack, then a checkpointed version that stores every k-th activation and recomputes the rest, with exact gradients and the √n memory trade-off.

**How to use this notebook**
1. Run the **Setup** cell.
2. For each task, fill in the `TODO` in the code cell, then run the **Tests** cell under it. You'll see ⏳ (not done yet), ✅ (passed) or ❌ (failed, with the reason).
3. Stuck? The **Reference solutions** at the bottom are collapsed. Try first.

Everything runs on CPU with NumPy, so no GPU is needed.

In [ ]:
# @title Setup: run this first
import numpy as np, math, time, itertools, collections
np.set_printoptions(precision=4, suppress=True)

def check(name, test, impl):
    """Run `test(impl)` and report: ⏳ not implemented, ✅ passed, ❌ failed with the reason."""
    try:
        test(impl)
        print(f"✅ {name}: all tests passed")
    except NotImplementedError:
        print(f"⏳ {name}: not implemented yet. Fill in the TODO in the cell above, then re-run this cell.")
    except AssertionError as e:
        print(f"❌ {name}: {e}")


def softmax(x, axis=-1):
    x = x - x.max(axis=axis, keepdims=True)
    e = np.exp(x)
    return e / e.sum(axis=axis, keepdims=True)

r = np.random.default_rng(0)
L, Wd = 16, 6
Ws = [r.normal(0, 0.5, (Wd, Wd)) for _ in range(L)]
x0 = r.normal(size=(3, Wd))

def layer(h, W):
    return np.tanh(h @ W)

def ref_grads(x):
    """Plain backprop of loss = sum(output) storing every activation."""
    acts = [x]
    for W in Ws:
        acts.append(layer(acts[-1], W))
    g = np.ones_like(acts[-1]); gW = [None] * L
    for i in range(L - 1, -1, -1):
        dz = g * (1 - acts[i + 1] ** 2)
        gW[i] = acts[i].T @ dz
        g = dz @ Ws[i].T
    return gW

## Task 1: Backward through one segment

Write the backward pass for layers `lo … hi−1` given the segment's **input** activation `h_lo` and the gradient `g` arriving at its output. Recompute the segment's activations from `h_lo` (that is the "recompute" part), then backpropagate:
$dz = g \odot (1 - a_{i+1}^2)$, $\;dW_i = a_i^\top dz$, $\;g \leftarrow dz\,W_i^\top$.
Return `(grads_dict {i: dW_i}, g_at_input, n_layer_evals)`.

In [ ]:
def backward_from(h_lo, lo, hi, g):
    # TODO: recompute acts for layers lo..hi-1 from h_lo, then backprop through them
    raise NotImplementedError

In [ ]:
# @title Tests for Task 1: run after implementing
def _test_backward_from(f):
    gW = ref_grads(x0)
    grads, g_in, evals = f(x0, 0, L, np.ones((3, Wd)))
    assert evals == L, "recomputing a segment of L layers takes L layer evaluations"
    assert all(np.allclose(grads[i], gW[i]) for i in range(L)), "gradients must match plain backprop"

check("backward_from", _test_backward_from, backward_from)

## Task 2: Checkpoint every k layers

Forward pass: keep only the activations at layers 0, k, 2k, … (the **checkpoints**), discard the rest.
Backward pass: for each segment, from last to first, recompute it from its checkpoint and backpropagate (use `backward_from`).
Return `(grads_list, peak_stored, extra_evals)`: `peak_stored` = number of checkpoints + the largest segment's activations held during its backward (k); `extra_evals` = layer evaluations spent on recomputation.

In [ ]:
def checkpointed_grads(x, k, bf=None):
    bf = bf or backward_from
    # TODO
    raise NotImplementedError

In [ ]:
# @title Tests for Task 2: run after implementing
def _test_checkpointed_grads(f):
    gW = ref_grads(x0)
    bf = backward_from_solution if f.__name__.endswith('_solution') else backward_from
    for k in [1, 2, 4, 8, 16]:
        grads, peak, extra = f(x0, k, bf)
        assert all(np.allclose(grads[i], gW[i]) for i in range(L)), f"k={k}: gradients must match"
        assert peak == L // k + k, f"k={k}: peak stored should be L/k + k = {L // k + k}, got {peak}"
        assert extra == L, f"k={k}: recomputation costs one extra forward (L layer evals), got {extra}"

check("checkpointed_grads", _test_checkpointed_grads, checkpointed_grads)

---
## Reference solutions
Try the tasks first. Each cell below is collapsed in Colab: double-click it (or click "Show code") to reveal the solution. Running it also checks the reference against the same tests.

In [ ]:
# @title Solution for Task 1: Backward through one segment
def backward_from_solution(h_lo, lo, hi, g):
    acts = [h_lo]
    for i in range(lo, hi):
        acts.append(layer(acts[-1], Ws[i]))
    grads = {}
    for j in range(hi - lo - 1, -1, -1):
        i = lo + j
        dz = g * (1 - acts[j + 1] ** 2)
        grads[i] = acts[j].T @ dz
        g = dz @ Ws[i].T
    return grads, g, hi - lo

check("backward_from (reference)", _test_backward_from, backward_from_solution)

In [ ]:
# @title Solution for Task 2: Checkpoint every k layers
def checkpointed_grads_solution(x, k, bf=None):
    bf = bf or backward_from_solution
    ckpt, h = {0: x}, x
    for i in range(L):
        h = layer(h, Ws[i])
        if (i + 1) % k == 0 and i + 1 < L:
            ckpt[i + 1] = h
    g = np.ones_like(h); grads = [None] * L; extra = 0
    for lo in sorted(ckpt, reverse=True):
        hi = min(lo + k, L)
        gd, g, ev = bf(ckpt[lo], lo, hi, g)
        for i, v in gd.items():
            grads[i] = v
        extra += ev
    return grads, len(ckpt) + k, extra

check("checkpointed_grads (reference)", _test_checkpointed_grads, checkpointed_grads_solution)

---
## Experiments
**The √n sweet spot.** Stored activations ≈ L/k + k, minimized at k = √L, for one extra forward pass. Chen et al. (2016) cut a 1,000-layer network from 48 GB to 7 GB for about 30% more time. Modern LLM training uses *selective* recomputation: recompute only the cheap-but-large attention-score part (Korthikanti et al.), saving most memory for little compute.

In [ ]:
for Lx in [16, 64, 256, 1024]:
    best = min(range(1, Lx + 1), key=lambda k: Lx / k + k)
    print(f"L={Lx:5d}: store everything = {Lx:5d} activations; checkpoint every √L = {best:3d} → {Lx // best + best:4d}  ({(Lx // best + best) / Lx:.1%} of the memory)")

---
## Takeaways
* Checkpointing trades compute for memory: store a few activations, recompute the rest during backward.
* Every k layers → memory ≈ L/k + k, best at k ≈ √L, cost ≈ one extra forward pass (~30% more time).
* Recompute is exact: gradients are identical.